# Notebook to open `xarray` datasets on SciServer-ceph and display basic information about them.

This code avoids OceanSpy.

If the datasets fail to open, make sure that your SciServer container includes the Poseidon (ceph) and Ocean Circulation (ceph) data volumes.

TWNH Jun '26

In [1]:
import time
import traceback
import intake
import pandas as pd
import os
import glob
import yaml

In [2]:
# Change this to the catalog you want to test
local_catalog_file1 = '/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xarray.yaml'

# If False: read one scalar from the first data variable only.
# If True: read one scalar from every data variable.
CHECK_ALL_VARS = False

In [3]:
def light_check(ds, check_all_vars=False):
    """
    Light integrity check for an xarray Dataset.

    This does not load the whole dataset. It only reads one scalar from one
    variable, or from every variable if check_all_vars=True.
    """

    print(f"    dims: {dict(ds.sizes)}")
    print(f"    data variables: {len(ds.data_vars)}")

    if len(ds.data_vars) == 0:
        print("    no data variables to test")
        return

    varnames = list(ds.data_vars) if check_all_vars else [list(ds.data_vars)[0]]

    for varname in varnames:
        da = ds[varname]

        print(f"    checking variable: {varname}, dims={da.dims}, shape={da.shape}")

        if da.size == 0:
            print("      skipping empty variable")
            continue

        indexer = {
            dim: 0
            for dim in da.dims
            if da.sizes.get(dim, 0) > 0
        }

        # Trigger a tiny actual read.
        da.isel(indexer).load()

        print("      ok")
        
def test_catalog_with_timing(catalog_path, check_all_vars=False):
    """
    Open every source in an Intake catalog, run a light xarray integrity check,
    and record timing and size information for each source.

    Size columns:
      - uncompressed_size_bytes: logical in-memory size from xarray Dataset.nbytes
      - compressed_size_bytes: physical size on disk from catalog urlpath/path
    """

    cat = intake.open_catalog(catalog_path)
    catalog_urlpaths = get_urlpaths_from_catalog_yaml(catalog_path)
    
    passed = []
    failed = []
    records = []

    for name in cat:
        print("\n" + "=" * 80)
        print(f"Testing source: {name}")
        print("=" * 80)

        ds = None
        total_t0 = time.perf_counter()

        open_time = None
        light_check_time = None
        uncompressed_size_bytes = None
        compressed_size_bytes = None
        compression_ratio = None
        source_urlpath = None
        error = ""
        status = "failed"

        try:
            source = cat[name]

            # Physical/compressed size from catalog path.
            source_urlpath = catalog_urlpaths.get(name)
            if source_urlpath is None:
                source_urlpath = get_source_urlpath(source)
                
            compressed_size_bytes = path_size(source_urlpath)

            if compressed_size_bytes is not None:
                print(f"Compressed/on-disk size: {format_bytes(compressed_size_bytes)}")
            else:
                print("Compressed/on-disk size: unknown")

            open_t0 = time.perf_counter()
            ds = source.to_dask()
            open_time = time.perf_counter() - open_t0

            print(f"Open time: {open_time:.1f} seconds")

            # Logical/uncompressed size from xarray.
            uncompressed_size_bytes = dataset_uncompressed_size(ds)

            if uncompressed_size_bytes is not None:
                print(f"Uncompressed/logical size: {format_bytes(uncompressed_size_bytes)}")
            else:
                print("Uncompressed/logical size: unknown")

            if (
                compressed_size_bytes is not None
                and compressed_size_bytes > 0
                and uncompressed_size_bytes is not None
            ):
                compression_ratio = uncompressed_size_bytes / compressed_size_bytes
                print(f"Logical/on-disk ratio: {compression_ratio:.2f}x")

            check_t0 = time.perf_counter()
            light_check(ds, check_all_vars=check_all_vars)
            light_check_time = time.perf_counter() - check_t0

            print(f"Light check time: {light_check_time:.1f} seconds")

            status = "passed"
            passed.append(name)

            print(f"PASS: {name}")

        except Exception as e:
            status = "failed"
            error = repr(e)

            print(f"FAIL: {name}")
            print(f"Error: {e}")
            traceback.print_exc()

            failed.append((name, error))

        finally:
            total_time = time.perf_counter() - total_t0

            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

            records.append(
                {
                    "source": name,
                    "status": status,
                    "open_time_s": open_time,
                    "light_check_time_s": light_check_time,
                    "total_time_s": total_time,
                    "compressed_size_bytes": compressed_size_bytes,
                    "compressed_size": format_bytes(compressed_size_bytes),
                    "uncompressed_size_bytes": uncompressed_size_bytes,
                    "uncompressed_size": format_bytes(uncompressed_size_bytes),
                    "logical_to_disk_ratio": compression_ratio,
                    "source_urlpath": source_urlpath,
                    "error": error,
                }
            )

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Catalog: {catalog_path}")
    print(f"Passed: {len(passed)}")
    print(f"Failed: {len(failed)}")

    if failed:
        print("\nFailed sources:")
        for name, error in failed:
            print(f"  - {name}: {error}")

    timing_results = pd.DataFrame(records)

    return passed, failed, timing_results

def format_bytes(n):
    """
    Human-readable byte formatter.
    """
    if n is None:
        return ""

    try:
        n = float(n)
    except Exception:
        return ""

    units = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    for unit in units:
        if abs(n) < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024

    return f"{n:.1f} EiB"

def directory_size(path):
    """
    Recursively compute directory size in bytes.
    """
    total = 0

    for root, dirs, files in os.walk(path):
        for fname in files:
            fpath = os.path.join(root, fname)
            try:
                total += os.path.getsize(fpath)
            except OSError:
                pass

    return total

def path_size(path_or_glob):
    """
    Compute size in bytes for a file, directory, glob, or list of paths/globs.
    """

    if path_or_glob is None:
        return None

    # Intake urlpath can sometimes be a list
    if isinstance(path_or_glob, (list, tuple)):
        total = 0
        any_found = False

        for item in path_or_glob:
            size = path_size(item)
            if size is not None:
                total += size
                any_found = True

        return total if any_found else None

    path_or_glob = str(path_or_glob)

    # Expand glob patterns
    matches = sorted(glob.glob(path_or_glob))

    if matches:
        total = 0
        for path in matches:
            if os.path.isdir(path):
                total += directory_size(path)
            elif os.path.isfile(path):
                total += os.path.getsize(path)
        return total

    # If no glob match, try literal path
    if os.path.isdir(path_or_glob):
        return directory_size(path_or_glob)

    if os.path.isfile(path_or_glob):
        return os.path.getsize(path_or_glob)

    return None

def dataset_uncompressed_size(ds):
    """
    Estimate uncompressed logical size of an xarray Dataset in bytes.
    """
    if ds is None:
        return None

    try:
        return int(ds.nbytes)
    except Exception:
        pass

    # Fallback
    try:
        total = 0
        for var in ds.variables.values():
            if hasattr(var, "nbytes"):
                total += int(var.nbytes)
        return total
    except Exception:
        return None

    import yaml

def get_urlpaths_from_catalog_yaml(catalog_path):
    """
    Read the Intake YAML directly and return a dict:
        {source_name: urlpath}
    """

    with open(catalog_path, "r") as f:
        catalog = yaml.safe_load(f)

    sources = catalog.get("sources", catalog)

    out = {}

    for name, entry in sources.items():
        if not isinstance(entry, dict):
            continue

        args = entry.get("args", {})
        if not isinstance(args, dict):
            continue

        for key in ["urlpath", "url", "path", "paths"]:
            if key in args:
                out[name] = args[key]
                break

    return out

In [4]:
passed, failed, results = test_catalog_with_timing(
    local_catalog_file1,
    check_all_vars=CHECK_ALL_VARS,
)


Testing source: grd_get_started
Compressed/on-disk size: 4.2 GiB
Open time: 6.1 seconds
Uncompressed/logical size: 4.2 GiB
Logical/on-disk ratio: 1.00x
    dims: {'Zp1': 217, 'Z': 216, 'Y': 880, 'X': 960, 'Xp1': 961, 'Yp1': 881, 'Zu': 216, 'Zl': 216}
    data variables: 30
    checking variable: drC, dims=('Zp1',), shape=(217,)
      ok
Light check time: 0.2 seconds
PASS: grd_get_started

Testing source: fld_get_started
Compressed/on-disk size: 44.6 GiB
Open time: 0.8 seconds
Uncompressed/logical size: 44.6 GiB
Logical/on-disk ratio: 1.00x
    dims: {'T': 4, 'Zd000001': 1, 'Y': 880, 'X': 960, 'Zmd000216': 216, 'Xp1': 961, 'Yp1': 881, 'Z': 216, 'Zl': 216}
    data variables: 49
    checking variable: EXFhs, dims=('T', 'Zd000001', 'Y', 'X'), shape=(4, 1, 880, 960)
      ok
Light check time: 0.0 seconds
PASS: fld_get_started

Testing source: avg_get_started
Compressed/on-disk size: 592.7 MiB
Open time: 0.5 seconds
Uncompressed/logical size: 592.7 MiB
Logical/on-disk ratio: 1.00x
    dims

/home/idies/mambaforge/envs/Oceanography/lib/python3.12/site-packages/intake/readers/readers.py:1334: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  return open_mfdataset(ofs, **kw)


Open time: 0.6 seconds
Uncompressed/logical size: 4.2 TiB
Logical/on-disk ratio: 2.17x
    dims: {'time': 9497, 'Zl': 50, 'face': 13, 'Y': 90, 'X': 90, 'Z': 50, 'Xp1': 90, 'Yp1': 90}
    data variables: 31
    checking variable: ADVr_SLT, dims=('time', 'Zl', 'face', 'Y', 'X'), shape=(9497, 50, 13, 90, 90)
      ok
Light check time: 0.6 seconds
PASS: daily_ecco_mean

Testing source: HYCOM_test
Compressed/on-disk size: 212.2 GiB
Open time: 0.0 seconds
Uncompressed/logical size: 1.2 TiB
Logical/on-disk ratio: 5.62x
    dims: {'time': 902, 'lat': 1251, 'lon': 701, 'Z': 101}
    data variables: 5
    checking variable: Eta, dims=('time', 'lat', 'lon'), shape=(902, 1251, 701)
      ok
Light check time: 0.6 seconds
PASS: HYCOM_test

Testing source: ETOPO
Compressed/on-disk size: 225.6 MiB
Open time: 0.0 seconds
Uncompressed/logical size: 1.7 GiB
Logical/on-disk ratio: 7.89x
    dims: {'Y': 10801, 'X': 21601}
    data variables: 1
    checking variable: Depth, dims=('Y', 'X'), shape=(10801, 21

In [5]:
summary_cols = [
    "source",
    "status",
    "open_time_s",
    "light_check_time_s",
    "total_time_s",
    "compressed_size",
    "uncompressed_size",
    "logical_to_disk_ratio",
    "error",
]

results[summary_cols]

,source,status,open_time_s,light_check_time_s,total_time_s,compressed_size,uncompressed_size,logical_to_disk_ratio,error
0,grd_get_started,passed,6.087412,0.189300,6.301266,4.2 GiB,4.2 GiB,0.999990,
1,fld_get_started,passed,0.803885,0.003668,0.812029,44.6 GiB,44.6 GiB,0.999995,
2,avg_get_started,passed,0.477290,0.003319,0.483355,592.7 MiB,592.7 MiB,0.999939,
3,grd_IGPwinter,passed,4.741920,0.849677,10.193897,2.5 TiB,9.2 TiB,3.650294,
4,fld_IGPwinter,passed,0.070513,0.050784,1.726001,2.5 TiB,9.2 TiB,3.650294,
5,grd_IGPyearlong,passed,4.227156,1.416777,31.401255,4.4 TiB,39.3 TiB,8.994989,
6,fld_IGPyearlong,passed,0.215118,0.014542,4.581379,4.4 TiB,39.3 TiB,8.994989,
7,grd_EGshelfIIseas2km_ERAI_6H,passed,3.305730,1.426025,14.211900,4.6 TiB,13.7 TiB,2.955836,
8,fld_EGshelfIIseas2km_ERAI_6H,passed,0.046785,0.009525,1.914416,4.6 TiB,13.7 TiB,2.955836,
9,grd_EGshelfIIseas2km_ERAI_1D,passed,2.969373,0.286835,3.334598,1014.0 MiB,8.9 GiB,8.960507,
